In [ ]:
# ==========================================
# 1. ALL REQUIRED IMPORTS
# ==========================================
import warnings

# Surgically mute MLflow's annoying Windows dataset source warning
warnings.filterwarnings(
    "ignore", 
    message="The specified dataset source can be interpreted in multiple ways"
)
import os
import ast
import pathlib
import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.data

from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ==========================================
# 2. THE EXPERIMENT TRACKING FUNCTION
# ==========================================
def run_experiment(run_name, model, X, y, mlb_encoder,
                   raw_dataframe=None,
                   dataset_source_name="cleaned_data",
                   use_department=False, 
                   max_features=10000, 
                   word_ngram_range=None, 
                   char_ngram_range=None,
                   min_df=1):
    """Dynamically builds, evaluates, and logs a multi-label classification pipeline."""
    with mlflow.start_run(run_name=run_name):
        print(f"--- Starting Experiment: {run_name} ---")
        
        if raw_dataframe is not None:
            print("Logging dataset metadata...")
            # Let pathlib safely generate a universally valid file URI for Windows
            source_uri = pathlib.Path(dataset_source_name).resolve().as_uri()
            
            mlflow_dataset = mlflow.data.from_pandas(
                raw_dataframe, 
                source=source_uri, 
                name="Retail_Commodity_Training_Data"
            )
            mlflow.log_input(mlflow_dataset, context="training")

        # Log Parameters
        mlflow.log_params({
            "use_department": use_department,
            "max_features": max_features,
            "word_ngram_range": word_ngram_range,
            "char_ngram_range": char_ngram_range,
            "min_df": min_df,
            "model_type": model.__class__.__name__
        })

        if word_ngram_range is None and char_ngram_range is None:
            print("No n-gram ranges provided")
            return
        
        # Build Preprocessor
        transformers = []
        if word_ngram_range:
            transformers.append(
                ('text', TfidfVectorizer(max_features=max_features, 
                                         ngram_range=word_ngram_range, 
                                         min_df=min_df), 'concatenated_text')
            )
        if char_ngram_range:
            transformers.append(
                ('char', TfidfVectorizer(max_features=max_features, 
                                         ngram_range=char_ngram_range, 
                                         analyzer='char_wb', 
                                         min_df=min_df), 'concatenated_text')
            )
        if use_department:
            transformers.append(
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), ['dpmt_num'])
            )
            
        preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

        # Build Pipeline
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('clf', OneVsRestClassifier(model))
        ])

        # -----------------------------------------------------------
        # Cross Validation & Out-of-Fold Predictions
        # -----------------------------------------------------------
        cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)
        print("Running 5-Fold Cross Validation...")
        
        # Keep this just to show the fold stability in the console
        cv_scores = cross_val_score(pipeline, X, y, cv=cv_strategy, scoring='f1_micro', n_jobs=-1)
        mean_f1 = cv_scores.mean()
        std_f1 = cv_scores.std() * 2
        
        print("Generating Out-of-Fold Predictions for Deep Evaluation...")
        y_pred_oof = cross_val_predict(pipeline, X, y, cv=cv_strategy, n_jobs=-1)
        
        # --- CALCULATE THE MULTI-LABEL METRICS ---
        from sklearn.metrics import f1_score, hamming_loss, accuracy_score
        
        metrics_dict = {
            "cv_f1_micro_mean": mean_f1,
            "cv_f1_micro_std": std_f1,
            "oof_f1_macro": f1_score(y, y_pred_oof, average='macro'),
            "oof_f1_weighted": f1_score(y, y_pred_oof, average='weighted'),
            "oof_hamming_loss": hamming_loss(y, y_pred_oof),
            "oof_exact_match_ratio": accuracy_score(y, y_pred_oof) # Scikit-learn's accuracy is Subset Accuracy for multi-label
        }
        
        # Log them all to MLflow in one single command!
        mlflow.log_metrics(metrics_dict)
        
        # Print them to the console so you can see them immediately
        print("\n--- Multi-Label Performance Metrics ---")
        for metric_name, value in metrics_dict.items():
            print(f"{metric_name}: {value:.4f}")
        print("---------------------------------------\n")

        print("Generating Out-of-Fold Predictions for Classification Report...")
        y_pred_oof = cross_val_predict(pipeline, X, y, cv=cv_strategy, n_jobs=-1)
        
        report = classification_report(
            y, 
            y_pred_oof, 
            target_names=mlb_encoder.classes_ 
        )
        print(report)

        with open("classification_report.txt", "w") as f:
            f.write(report)
        mlflow.log_artifact("classification_report.txt")

        # -----------------------------------------------------------
        # Train & Save Final Production Model
        # -----------------------------------------------------------
        print("Training final model on full dataset to log artifacts...")
        pipeline.fit(X, y)
        
        # WARNING 2 FIXED: Use secure 'skops' serialization
        mlflow.sklearn.log_model(
            pipeline, 
            "model", 
            serialization_format="skops"
        )
        
        joblib.dump(mlb_encoder, "mlb_encoder.joblib")
        mlflow.log_artifact("mlb_encoder.joblib")
        
        print("Experiment successfully logged!\n")

In [54]:
# ==========================================
# 3. DATA LOADING & PREPARATION
# ==========================================
data_path = "../data/processed/combined_commodity_data.csv"
df = pd.read_csv(data_path)

# Safe string splitting
df['commodity'] = df['commodity'].astype(str).apply(lambda x: [item.strip() for item in x.split(',')])

X_text_only = df[['concatenated_text']]
log_df_text_only = df[['concatenated_text', 'commodity']]

X_text_and_dept = df[['concatenated_text', 'dpmt_num']]
log_df_text_and_dept = df[['concatenated_text', 'dpmt_num', 'commodity']]

mlb = MultiLabelBinarizer()
multi_hot_labels = mlb.fit_transform(df['commodity'])
y_full = multi_hot_labels

In [55]:
# ==========================================
# 4. RUN THE EXPERIMENTS
# ==========================================
tracking_uri = pathlib.Path("../mlruns").resolve().as_uri()
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("Commodity_Labels")

# Your Champion LinearSVC Baseline
run_experiment(
    run_name="LinearSVC_Words",
    model=LinearSVC(class_weight='balanced', random_state=42),
    X=X_text_only,             
    y=y_full, 
    mlb_encoder=mlb,
    use_department=False,
    raw_dataframe=log_df_text_only, 
    dataset_source_name=data_path,
    word_ngram_range=(1, 2)
)

--- Starting Experiment: LinearSVC_Words ---
Logging dataset metadata...
Running 5-Fold Cross Validation...


c:\Users\cjchen\Desktop\personal-projects\commodity-app\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


Generating Out-of-Fold Predictions for Deep Evaluation...

--- Multi-Label Performance Metrics ---
cv_f1_micro_mean: 0.9650
cv_f1_micro_std: 0.0067
oof_f1_macro: 0.9447
oof_f1_weighted: 0.9708
oof_hamming_loss: 0.0052
oof_exact_match_ratio: 0.9356
---------------------------------------

Generating Out-of-Fold Predictions for Classification Report...


c:\Users\cjchen\Desktop\personal-projects\commodity-app\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

        BEEF       0.99      0.99      0.99      1111
       COCOA       0.87      0.99      0.93      2556
      COFFEE       0.98      0.99      0.99       519
       DAIRY       0.97      0.99      0.98       356
        EGGS       1.00      0.99      1.00       187
 ENERGY STAR       1.00      1.00      1.00      2722
       FIBER       1.00      1.00      1.00      6496
        LAMB       0.86      0.95      0.90        20
       OTHER       0.80      1.00      0.89         4
    PALM OIL       0.51      0.89      0.65      1067
        PORK       0.96      0.99      0.97       294
     POULTRY       0.97      0.99      0.98       643
     SEAFOOD       1.00      1.00      1.00      6102
         SOY       0.96      0.94      0.95        51

   micro avg       0.94      0.99      0.96     22128
   macro avg       0.92      0.98      0.94     22128
weighted avg       0.96      0.99      0.97     22128
 samples avg       0.96   

2026/03/11 16:41:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 16:42:03 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\cjchen\AppData\Local\Temp\1\tmpzquv7xd1\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/11 16:42:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Experiment successfully logged!

